# Phase 3 - Notebook 01: Vision Transformer (ViT) Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/01_vit_foundations.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand Vision Transformer (ViT) architecture fundamentals
2. Learn how images are tokenized into patches
3. Understand self-attention and positional encoding
4. Implement a simplified ViT forward pass
5. See how ViT is used as the encoder backbone in DUSt3R

**Estimated Time**: 45 minutes

**Prerequisites**: Basic PyTorch knowledge, linear algebra

---

## Context

DUSt3R uses a Vision Transformer (ViT) as the **encoder** to extract features from input images.
Instead of CNN convolutional layers, ViT operates on image patches and uses self-attention.

**Why ViT over CNN for DUSt3R?**
- ViT has a larger receptive field (attends to all pixels simultaneously)
- Better for capturing global geometric relationships
- More parameter-efficient for dense prediction tasks


## 0. Setup

In [ ]:
# Setup
import sys
sys.path.insert(0, '../..')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Patch Embedding: 从图像到Token

ViT的第一步：将图像分割成固定大小的补丁（patches），然后将每个补丁线性投影到嵌入维度。

```
Image (H×W×3)  →  Patches (L×P²×3)  →  Embeddings (L×D)
其中:
  L = (H/P) × (W/P) = 补丁数量
  P = 补丁大小 (通常16)
  D = 嵌入维度 (通常768或1024)
```

**示例**：一张 224×224 的图像，补丁大小=16
- 补丁数量 = (224/16) × (224/16) = 14 × 14 = 196
- 每个补丁大小 = 16×16×3 = 768 维
- 线性投影 → 768 维嵌入


In [ ]:
class PatchEmbedding(nn.Module):
    """将图像分割为补丁并投影到嵌入空间。"""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_dim = patch_size * patch_size * in_channels
        
        # 线性投影补丁（可用Conv2d实现）
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
        
    def forward(self, x):
        # x: [B, C, H, W]
        # 输出: [B, L, D]
        x = self.proj(x)  # [B, D, H/P, W/P]
        x = x.flatten(2)  # [B, D, L]
        x = x.transpose(1, 2)  # [B, L, D]
        return x

# 测试
batch_size = 2
img_size = 224
patch_size = 16
embed_dim = 768

patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
x_img = torch.randn(batch_size, 3, img_size, img_size)
x_patches = patch_embed(x_img)

print(f"输入图像: {x_img.shape}")
print(f"补丁嵌入: {x_patches.shape}")
print(f"补丁数量: {patch_embed.num_patches}")
print(f"嵌入维度: {embed_dim}")

## 2. Positional Encoding: 位置信息

Transformer 本身没有空间结构的概念（不像 CNN），因此需要显式添加补丁位置信息。

ViT 使用**可学习的位置编码**：每个补丁位置 (i, j) 对应一个 D 维向量。


In [ ]:
class PositionalEncoding(nn.Module):
    """可学习的位置编码。"""
    
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        # 可学习的位置编码表
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim))
        # 初始化为接近0（使用较小的标准差）
        nn.init.normal_(self.pos_embed, std=0.02)
    
    def forward(self, x):
        # x: [B, L, D]
        return x + self.pos_embed

# 测试
num_patches = patch_embed.num_patches
pos_encoding = PositionalEncoding(num_patches, embed_dim)
x_with_pos = pos_encoding(x_patches)

print(f"带位置编码的补丁: {x_with_pos.shape}")
print(f"位置编码参数: {pos_encoding.pos_embed.shape}")

## 3. Self-Attention: 补丁之间的交互

**Self-Attention 机制** 允许每个补丁与所有其他补丁进行交互。

公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- Q (Query) = x @ W_q
- K (Key) = x @ W_k
- V (Value) = x @ W_v
- $\sqrt{d_k}$ 是缩放因子（防止梯度消失）

**Multi-Head Attention**：并行运行多个注意力头，捕捉不同的几何模式。


In [ ]:
class MultiHeadAttention(nn.Module):
    """多头自注意力。"""
    
    def __init__(self, embed_dim, num_heads=8, attn_drop=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        assert embed_dim % num_heads == 0, "embed_dim 必须被 num_heads 整除"
        
        # 投影矩阵
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_drop)
    
    def forward(self, x):
        # x: [B, L, D]
        B, L, D = x.shape
        
        # 投影为 Q, K, V
        qkv = self.qkv(x)  # [B, L, 3D]
        qkv = qkv.reshape(B, L, 3, self.num_heads, self.head_dim)  # [B, L, 3, H, D/H]
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, H, L, D/H]
        
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # 计算注意力权重
        attn = (q @ k.transpose(-2, -1)) * self.scale  # [B, H, L, L]
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        # 应用注意力权重到值
        x_attn = attn @ v  # [B, H, L, D/H]
        x_attn = x_attn.transpose(1, 2).reshape(B, L, D)  # [B, L, D]
        
        # 输出投影
        x_out = self.proj(x_attn)
        
        return x_out, attn

# 测试
mha = MultiHeadAttention(embed_dim, num_heads=12)
x_attn, attn_weights = mha(x_with_pos)

print(f"注意力输出: {x_attn.shape}")
print(f"注意力权重: {attn_weights.shape}")
print(f"\n注意力权重信息：")
print(f"  - 形状: [batch={attn_weights.shape[0]}, heads={attn_weights.shape[1]}, " +
      f"query_len={attn_weights.shape[2]}, key_len={attn_weights.shape[3]}]")
print(f"  - 范围: [{attn_weights.min():.4f}, {attn_weights.max():.4f}]")

## 4. Transformer Block: MLP + LayerNorm

一个完整的 Transformer 块包含：
1. **LayerNorm** → 标准化
2. **MultiHeadAttention** → 补丁交互
3. **MLP** (2 层全连接) → 特征变换
4. **残差连接** → 梯度流


In [ ]:
class TransformerBlock(nn.Module):
    """一个 Transformer 块。"""
    
    def __init__(self, embed_dim, num_heads=8, mlp_dim=None, dropout=0.0):
        super().__init__()
        if mlp_dim is None:
            mlp_dim = embed_dim * 4  # 标准配置
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, attn_drop=dropout)
        
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        # Attention with residual
        x_norm = self.norm1(x)
        x_attn, _ = self.attn(x_norm)
        x = x + x_attn  # 残差连接
        
        # MLP with residual
        x_norm = self.norm2(x)
        x_mlp = self.mlp(x_norm)
        x = x + x_mlp  # 残差连接
        
        return x

# 测试
block = TransformerBlock(embed_dim, num_heads=12)
x_block = block(x_with_pos)

print(f"Transformer 块输出: {x_block.shape}")

## 5. Complete ViT Encoder

堆叠多个 Transformer 块来构建完整的编码器。


In [ ]:
class ViTEncoder(nn.Module):
    """Vision Transformer 编码器。"""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, num_heads=12, num_layers=12, mlp_dim=3072):
        super().__init__()
        
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.pos_encoding = PositionalEncoding(self.patch_embed.num_patches, embed_dim)
        
        # 堆叠 Transformer 块
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_dim)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        # x: [B, C, H, W]
        x = self.patch_embed(x)  # [B, L, D]
        x = self.pos_encoding(x)
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        # 返回: [B, L, D]
        return x

# 测试小规模 ViT
vit = ViTEncoder(
    img_size=224, patch_size=16, in_channels=3,
    embed_dim=768, num_heads=12, num_layers=2,  # 2 层用于演示
    mlp_dim=3072
)

x_test = torch.randn(1, 3, 224, 224)
features = vit(x_test)

print(f"输入图像: {x_test.shape}")
print(f"编码器输出特征: {features.shape}")
print(f"参数数量: {sum(p.numel() for p in vit.parameters()) / 1e6:.1f}M")

## 6. 可视化注意力权重

注意力权重显示每个补丁关注哪些其他补丁。


In [ ]:
# 创建一个简单的彩色图像用于可视化
np.random.seed(42)
img = np.zeros((224, 224, 3))
# 绘制几个几何形状
img[50:100, 50:100] = [1, 0, 0]  # 红色正方形
img[150:180, 150:180] = [0, 1, 0]  # 绿色正方形
img[100:150, 120:170] = [0, 0, 1]  # 蓝色矩形

# 转换为张量
img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0)

# 前向传播
with torch.no_grad():
    # 获取第一个块的注意力权重
    x = vit.patch_embed(img_tensor)
    x = vit.pos_encoding(x)
    x_after_block0 = vit.blocks[0](x)
    
    # 手动获取注意力权重
    x_norm = vit.blocks[0].norm1(x)
    _, attn_weights = vit.blocks[0].attn(x_norm)
    # attn_weights: [B, H, L, L]

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 原始图像
ax = axes[0, 0]
ax.imshow(img)
ax.set_title('输入合成图像')
ax.axis('off')

# 补丁网格
ax = axes[0, 1]
ax.imshow(img)
# 绘制补丁网格
for i in range(0, 224, 16):
    ax.axhline(i, color='gray', alpha=0.3, linewidth=0.5)
    ax.axvline(i, color='gray', alpha=0.3, linewidth=0.5)
ax.set_title('补丁分割 (16×16)')
ax.axis('off')

# 中心补丁的注意力图
ax = axes[1, 0]
center_patch_idx = 7 * 14 + 7  # 中央补丁
attn_map = attn_weights[0, 0, center_patch_idx, :].cpu().numpy()  # 第一个注意力头
attn_map_2d = attn_map.reshape(14, 14)
im = ax.imshow(attn_map_2d, cmap='hot')
ax.set_title(f'中心补丁的注意力权重 (第1头)')
ax.set_xlabel('补丁列')
ax.set_ylabel('补丁行')
plt.colorbar(im, ax=ax)

# 平均注意力
ax = axes[1, 1]
mean_attn = attn_weights[0].mean(dim=0)  # 平均所有头
mean_attn_center = mean_attn[center_patch_idx, :].cpu().numpy().reshape(14, 14)
im = ax.imshow(mean_attn_center, cmap='hot')
ax.set_title(f'中心补丁的平均注意力权重 (所有头)')
ax.set_xlabel('补丁列')
ax.set_ylabel('补丁行')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('vit_attention.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"注意力权重统计：")
print(f"  - 最大值: {attn_weights.max():.4f}")
print(f"  - 最小值: {attn_weights.min():.4f}")
print(f"  - 平均值: {attn_weights.mean():.4f}")

## 7. DUSt3R 中的 ViT

DUSt3R 使用 ViT 作为编码器来处理图像对：

```
Image 1 (H×W×3)   Image 2 (H×W×3)
      ↓                  ↓
  ViT Encoder      ViT Encoder (共享权重)
      ↓                  ↓
  Features 1 (N×D)  Features 2 (N×D)
      \                /
       → Cross-Attention Decoder
         (在两个特征图之间交互)
           ↓
         Pointmaps
```

**关键点**：
- 编码器处理每张图像，提取 patch-wise 特征
- 特征维度通常是 768 或 1024 (embed_dim)
- 跨视图解码器使用这些特征进行交互
- 最终预测 3D 点坐标（Pointmap）


In [ ]:
# DUSt3R 编码器的概念演示
print("DUSt3R 编码器流程示例：")
print()
print("Input:")
print(f"  Image 1: [B=1, C=3, H=512, W=512]")
print(f"  Image 2: [B=1, C=3, H=512, W=512]")
print()

img_size = 512
patch_size = 16
embed_dim = 1024
num_patches = (img_size // patch_size) ** 2

print(f"Patch Embedding (patch_size={patch_size}, embed_dim={embed_dim}):")
print(f"  Image → Patches: [1, 3, 512, 512] → [1, {num_patches}, {embed_dim}]")
print()

print(f"ViT Encoder (共享权重):")
print(f"  Patches + Positional Encoding → {num_patches} Transformer Blocks")
print(f"  Output: [1, {num_patches}, {embed_dim}]")
print()

print(f"两个编码器输出：")
print(f"  Features 1: [1, {num_patches}, {embed_dim}]")
print(f"  Features 2: [1, {num_patches}, {embed_dim}]")
print()

print(f"Cross-Attention Decoder:")
print(f"  Input: Features1 + Features2")
print(f"  Output: Pointmaps (每个像素3D坐标)")
print(f"  Output shape: [1, H=512, W=512, 3]")

## 8. Summary

**Key Takeaways:**

1. **Patch Embedding**: 将图像分割成固定大小的补丁，线性投影到嵌入空间
   - 这消除了 CNN 的局部偏差，允许全局交互

2. **Positional Encoding**: 添加可学习的位置信息
   - 使 Transformer 能够感知空间结构

3. **Self-Attention**: 每个补丁与所有其他补丁交互
   - 多头注意力捕捉不同的几何关系

4. **Transformer Block**: 注意力 + MLP + 残差连接
   - LayerNorm 稳定训练

5. **ViT 编码器**: 堆叠 12-24 个 Transformer 块
   - 每个块进一步细化特征

6. **在 DUSt3R 中的使用**:
   - ViT 提取图像特征
   - 交叉注意力解码器进行视图间交互
   - 最终预测 Pointmaps (3D 点云)

---

**下一步**: [02_pointmap_representation.ipynb](./02_pointmap_representation.ipynb) — 理解 Pointmap 表示与深度图的区别
